<a href="https://colab.research.google.com/github/RAHUL-REDDY-A/ML/blob/main/ML_LAB_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RAHUL REDDY \
**BL.EN.U4CSE23102**

A1. Implement stacking classifier or regressor (depending on your project problem). The base models should be the list of classifiers / regressors already implemented. Experiment with various metamodels (final_estimator). Use above references 1 & 2.


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              StackingClassifier, ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

# Load & Preprocess
def load_data(filepath):
    df = pd.read_csv(filepath).head(1000)   # limit to 1000 rows
    df['Mental_Health_Condition'] = df['Mental_Health_Condition'].map({'Yes': 1, 'No': 0})
    X = df.drop(['Mental_Health_Condition', 'User_ID'], axis=1)
    y = df['Mental_Health_Condition']
    return X, y

def create_preprocessor():
    num_cols = ['Age', 'Sleep_Hours', 'Work_Hours',
                'Physical_Activity_Hours', 'Social_Media_Usage']
    cat_cols = ['Gender', 'Occupation', 'Country', 'Severity',
                'Consultation_History', 'Stress_Level', 'Diet_Quality',
                'Smoking_Habit', 'Alcohol_Consumption', 'Medication_Usage']

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

def create_models():
    base_models = [
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('svm', SVC(probability=True, random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('nb', GaussianNB()),
        ('et', ExtraTreesClassifier(n_estimators=100, random_state=42))
    ]
    meta = LogisticRegression(max_iter=1000, random_state=42)
    return StackingClassifier(estimators=base_models, final_estimator=meta, cv=5, n_jobs=-1)

# Main
if __name__ == "__main__":
    X, y = load_data("MHDS.csv")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    preprocessor = create_preprocessor()
    model = create_models()
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    print("A1 Accuracy:", accuracy_score(y_test, preds))


A1 Accuracy: 0.5366666666666666


A2. Implement pipeline to allow multiple steps of data processing and classification to be executed simultaneously. Use reference 3 above for pipeline construction and execution.


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Load & Preprocess
def load_data(filepath):
    df = pd.read_csv(filepath).head(1000)
    df['Mental_Health_Condition'] = df['Mental_Health_Condition'].map({'Yes': 1, 'No': 0})
    X = df.drop(['Mental_Health_Condition', 'User_ID'], axis=1)
    y = df['Mental_Health_Condition']
    return X, y

def create_preprocessor():
    num_cols = ['Age', 'Sleep_Hours', 'Work_Hours',
                'Physical_Activity_Hours', 'Social_Media_Usage']
    cat_cols = ['Gender', 'Occupation', 'Country', 'Severity',
                'Consultation_History', 'Stress_Level', 'Diet_Quality',
                'Smoking_Habit', 'Alcohol_Consumption', 'Medication_Usage']

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

# Main
if __name__ == "__main__":
    X, y = load_data("MHDS.csv")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    preprocessor = create_preprocessor()
    model = LogisticRegression(max_iter=1000, random_state=42)
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted')
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

    print("A2 Logistic Regression Pipeline")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
    print(f"CV Mean: {cv.mean():.4f} ± {cv.std():.4f}")


A2 Logistic Regression Pipeline
Accuracy: 0.4800
Precision: 0.4801, Recall: 0.4800, F1: 0.4797
CV Mean: 0.4971 ± 0.0539


A3. Using LIME explainer, explain the outcomes of pipeline.

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from lime import lime_tabular

# Load & Preprocess
def load_data(filepath):
    df = pd.read_csv(filepath).head(1000)
    df['Mental_Health_Condition'] = df['Mental_Health_Condition'].map({'Yes': 1, 'No': 0})
    X = df.drop(['Mental_Health_Condition', 'User_ID'], axis=1)
    y = df['Mental_Health_Condition']
    return X, y

def create_preprocessor():
    num_cols = ['Age', 'Sleep_Hours', 'Work_Hours',
                'Physical_Activity_Hours', 'Social_Media_Usage']
    cat_cols = ['Gender', 'Occupation', 'Country', 'Severity',
                'Consultation_History', 'Stress_Level', 'Diet_Quality',
                'Smoking_Habit', 'Alcohol_Consumption', 'Medication_Usage']

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

# Main
if __name__ == "__main__":
    X, y = load_data("MHDS.csv")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    preprocessor = create_preprocessor()
    model = LogisticRegression(max_iter=1000, random_state=42)
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])
    pipe.fit(X_train, y_train)

    # Build LIME explainer
    X_train_proc = pipe.named_steps['pre'].fit_transform(X_train)
    feature_names = pipe.named_steps['pre'].get_feature_names_out()

    explainer = lime_tabular.LimeTabularExplainer(
        X_train_proc,
        feature_names=feature_names,
        class_names=['No', 'Yes'],
        mode='classification'
    )

    X_test_proc = pipe.named_steps['pre'].transform(X_test)
    exp = explainer.explain_instance(
        X_test_proc[0],
        pipe.named_steps['clf'].predict_proba,
        num_features=5
    )

    print("A3 LIME Explanation for instance 0:")
    for feat, weight in exp.as_list():
        print(f"{feat}: {weight:.4f}")


A3 LIME Explanation for instance 0:
cat__Country_USA <= 0.00: -0.1331
cat__Country_UK <= 0.00: -0.1108
cat__Country_Other > 0.00: -0.1031
cat__Country_Canada <= 0.00: -0.0972
cat__Diet_Quality_Unhealthy <= 0.00: -0.0576
